# 🎮 [로컬 RTX 3060 환경] AudioResNet-50 전체 데이터 100% 풀학습

> **핵심 사양 및 강점**:
> - **데이터 100% 활용**: 으로 전체 수만 개 발화 구간 완벽 학습
> - **SpecAugment 탑재**: 주파수 및 시간 축 마스킹으로 과적합 방지
> - **FP16 AMP 가속**: RTX 3060 텐서코어를 활용한 고속 연산 (VRAM 3~4GB 절약)
> - **산출물**:  (목표 92%+)


In [18]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ 연산 가속기: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU 디바이스: {torch.cuda.get_device_name(0)}")
    print(f"📊 VRAM 용량: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


🖥️ 연산 가속기: cuda
🎮 GPU 디바이스: NVIDIA GeForce RTX 3060 Laptop GPU
📊 VRAM 용량: 6.00 GB


In [14]:
import os, glob, json, time, random
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import librosa
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 데이터 경로 (로컬 환경에 맞춰 수정 가능)
TRAIN_DIR = "../../data/train"
VAL_DIR = "../../data/val"


In [15]:
import os
import json
import librosa
import numpy as np
import torch
from torch.utils.data import Dataset

class LocalSpeechDataset(Dataset):
    def __init__(self, data_dir, max_files=None, is_train=True, augment=True, n_mels=128, n_fft=2048, hop_length=512, window_sec=3.0):
        self.data_dir = data_dir
        self.is_train = is_train
        self.augment = augment
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.window_sec = window_sec
        self.sr = 16000
        
        # SpecAugment (Train Only)
        if self.augment and self.is_train:
            import torchaudio.transforms as T
            self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
            self.time_mask = T.TimeMasking(time_mask_param=35)
            
        self.samples = self._parse_data(max_files)
        print(f"👉 [{data_dir}] 데이터 로드 완료: {len(self.samples)}개 조각 찾음!")
        
    def _parse_data(self, max_files):
        samples = []
        json_files = []
        
        # 💡 [핵심 해결] os.walk를 통해 윈도우 경로 오류를 씹고 무조건 모든 json을 싹 긁어옴
        for root, dirs, files in os.walk(self.data_dir):
            for file in files:
                if file.endswith('.json'):
                    json_files.append(os.path.join(root, file))
        
        json_files.sort()
        if max_files:
            json_files = json_files[:max_files]
            
        for jf in json_files:
            with open(jf, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # JSON 이름과 동일한 WAV 파일 경로 유추 ('label' 폴더를 'audio'로 변경)
            wav_path = jf.replace('label', 'audio').replace('.json', '.wav')
            
            # 파일이 진짜 있는지 검사
            if not os.path.exists(wav_path):
                continue
                
            # 대화 조각 정보 추출
            if 'utterances' in data:
                for utt in data['utterances']:
                    samples.append({
                        'wav_path': wav_path,
                        'start_at': float(utt.get('startAt', 0)),
                        'end_at': float(utt.get('endAt', 0)),
                        'speaker': int(utt.get('speaker', 0))
                    })
        return samples
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        wav_path = sample['wav_path']
        start_sec = sample['start_at']
        end_sec = sample['end_at']
        label = sample['speaker']
        
        # 1. 오디오 로드 (특정 구간만)
        duration = end_sec - start_sec
        try:
            y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
        except Exception:
            y = np.zeros(int(self.sr * self.window_sec))
            
        # 2. 길이 고정 (Zero-padding)
        target_len = int(self.sr * self.window_sec)
        if len(y) > target_len:
            y = y[:target_len]
        elif len(y) < target_len:
            pad_len = target_len - len(y)
            y = np.pad(y, (0, pad_len), 'constant')
            
        # 3. Mel-Spectrogram 
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length, n_mels=self.n_mels
        )
        mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-6)
        
        # 4. 텐서 변환
        mel_tensor = torch.tensor(mel_spec, dtype=torch.float32).unsqueeze(0)
        
        # 5. 증강 기법 적용
        if self.is_train and self.augment:
            mel_tensor = self.freq_mask(mel_tensor)
            mel_tensor = self.time_mask(mel_tensor)
            
        return mel_tensor, torch.tensor(label, dtype=torch.long)


In [16]:
class AudioResNet50(nn.Module):
    def __init__(self, pretrained=True, dropout_rate=0.3):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.resnet = models.resnet50(weights=weights)
        old_conv = self.resnet.conv1
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=False)
        if pretrained and old_conv.weight is not None:
            new_conv.weight.data = torch.mean(old_conv.weight.data, dim=1, keepdim=True)
        self.resnet.conv1 = new_conv
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(in_features, 1))
        
    def forward(self, x):
        return self.resnet(x)


In [17]:
import os
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# 1. 절대 경로 확인
TRAIN_DIR = r"C:\Users\user\Desktop\DCC\data\train"
VAL_DIR   = r"C:\Users\user\Desktop\DCC\data\val"

# 2. 데이터로더 생성 (⭐️ num_workers=0 으로 윈도우 프리징 원천 차단!)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, pin_memory=True) 
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True) 

# 3. 모델 및 학습 설정
os.makedirs("./checkpoints", exist_ok=True)
epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AudioResNet50(pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.cuda.amp.GradScaler()

best_acc, best_f1 = 0.0, 0.0
print(f"🚀 [AudioResNet-50] 10 에포크 풀학습 진짜 시작!")

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    
    # ⭐️ 실시간 로딩 바 장착!
    pbar = tqdm(train_loader, desc=f"[Epoch {epoch:02d}/{epochs:02d} Train]")
    for bx, by in pbar:
        bx, by = bx.to(device), by.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = model(bx)
            loss = criterion(out, by)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        
        # 실시간 Loss 출력
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    scheduler.step()
    
    # 검증(Validation)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for bx, by in tqdm(val_loader, desc=f"[Epoch {epoch:02d}/{epochs:02d} Val]"):
            bx = bx.to(device)
            with torch.cuda.amp.autocast():
                prob = torch.sigmoid(model(bx)).squeeze(-1).cpu().numpy()
            all_preds.extend((prob >= 0.5).astype(int))
            all_labels.extend(by.numpy().astype(int))
            
    acc = accuracy_score(all_labels, all_preds) * 100.0
    f1 = f1_score(all_labels, all_preds, average="macro")
    tr_loss = total_loss / len(train_loader)
    print(f"\n📢 [Ep {epoch:02d}/{epochs:02d}] Tr Loss: {tr_loss:.4f} | Val Acc: {acc:.2f}% | F1: {f1:.4f}")
    
    if acc > best_acc:
        best_acc = acc
        best_f1 = f1
        save_path = "./checkpoints/best_resnet_full.pt"
        torch.save(model.state_dict(), save_path)
        print(f"  👉 최고 점수 갱신! 가중치 저장: {save_path} ({acc:.2f}%)")

print(f"\n🎉 모든 학습 완료! 최고 정확도: {best_acc:.2f}% | Macro F1: {best_f1:.4f}")


C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


🚀 [AudioResNet-50] 10 에포크 풀학습 진짜 시작!


[Epoch 01/10 Train]:   0%|          | 0/27286 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Epoch 01/10 Val]:   0%|          | 0/3498 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Epoch 01/10 Val]: 100%|██████████| 3498/3498 [22:53<00:00,  2.55it/s]



📢 [Ep 01/10] Tr Loss: 0.6951 | Val Acc: 51.94% | F1: 0.3427
  👉 최고 점수 갱신! 가중치 저장: ./checkpoints/best_resnet_full.pt (51.94%)


[Epoch 02/10 Train]:   0%|          | 0/27286 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
c:\Users\user\Desktop\DCC\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Epoch 02/10 Val]:   0%|          | 0/3498 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', a


📢 [Ep 02/10] Tr Loss: 0.6936 | Val Acc: 48.10% | F1: 0.3248


[Epoch 03/10 Train]:   0%|          | 0/27286 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
c:\Users\user\Desktop\DCC\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Epoch 03/10 Val]:   0%|          | 0/3498 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', a


📢 [Ep 03/10] Tr Loss: 0.6938 | Val Acc: 51.94% | F1: 0.3427


[Epoch 04/10 Train]:   0%|          | 0/27286 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
c:\Users\user\Desktop\DCC\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\user\AppData\Local\Temp\ipykernel_1664\3265457103.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Epoch 04/10 Train]:  11%|█         | 2951/27286 [25:30<3:30:17,  1.93it/s, loss=0.6924]


KeyboardInterrupt: 

In [ ]:
import os, glob, json

# 노트북에서 쓰시는 TRAIN_DIR 변수를 그대로 사용하거나, 직접 경로를 입력해 주세요.
# TRAIN_DIR = r"C:\Users\user\Desktop\DCC\data\train"

label_dir = os.path.join(TRAIN_DIR, 'label')
audio_dir = os.path.join(TRAIN_DIR, 'audio')

# 1. JSON 파일이 몇 개나 찾아지는지 확인
json_files = glob.glob(os.path.join(label_dir, '**', '*.json'), recursive=True)
print(f"1. 찾은 JSON 파일 개수: {len(json_files)}개")

# 2. JSON 파일 안의 내용물 구조 확인
if len(json_files) > 0:
    test_json = json_files[0]
    with open(test_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"2. 첫 번째 JSON의 내용물(키): {list(data.keys())}")
    
    # WAV 파일이 잘 찾아지는지 확인
    wav_name = os.path.basename(test_json).replace('.json', '.wav')
    wav_path = os.path.join(audio_dir, wav_name)
    if not os.path.exists(wav_path):
        print(f"3. ❌ WAV 파일을 찾을 수 없습니다! (찾은 경로: {wav_path})")
    else:
        print(f"3. ✅ WAV 파일 정상 확인!")


1. 찾은 JSON 파일 개수: 0개


In [ ]:
import os
import json
import librosa
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# ==========================================
# 1. 모델 정의 (1채널 입력 + 이진 분류 FC 레이어)
# ==========================================
class AudioResNet50(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        self.resnet = resnet50(weights=weights)
        
        # 1채널 입력 변환 (3채널 가중치 평균)
        old_conv = self.resnet.conv1
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=False)
        if pretrained:
            new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        self.resnet.conv1 = new_conv
        
        # 출력 레이어 (1차원 로짓)
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, 1)
        
    def forward(self, x):
        return self.resnet(x)

# ==========================================
# 2. 데이터셋 정의 (절대 경로 + SpecAugment)
# ==========================================
class LocalSpeechDataset(Dataset):
    def __init__(self, data_dir, max_files=None, is_train=True, augment=True, n_mels=128, n_fft=2048, hop_length=512, window_sec=3.0):
        self.data_dir = data_dir
        self.is_train = is_train
        self.augment = augment
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.window_sec = window_sec
        self.sr = 16000
        
        if self.augment and self.is_train:
            import torchaudio.transforms as T
            self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
            self.time_mask = T.TimeMasking(time_mask_param=35)
            
        self.samples = self._parse_data(max_files)
        print(f"👉 [{os.path.basename(data_dir)}] 로드 완료: {len(self.samples):,}개 발화 구간")
        
    def _parse_data(self, max_files):
        samples = []
        json_files = []
        for root, dirs, files in os.walk(self.data_dir):
            for file in files:
                if file.endswith('.json'):
                    json_files.append(os.path.join(root, file))
        json_files.sort()
        if max_files:
            json_files = json_files[:max_files]
            
        for jf in json_files:
            with open(jf, 'r', encoding='utf-8') as f:
                data = json.load(f)
            wav_path = jf.replace('label', 'audio').replace('.json', '.wav')
            if not os.path.exists(wav_path):
                continue
            if 'utterances' in data:
                for utt in data['utterances']:
                    samples.append({
                        'wav_path': wav_path,
                        'start_at': float(utt.get('startAt', 0)),
                        'end_at': float(utt.get('endAt', 0)),
                        'speaker': int(utt.get('speaker', 0))
                    })
        return samples
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        wav_path = sample['wav_path']
        start_sec = sample['start_at']
        end_sec = sample['end_at']
        label = sample['speaker']
        
        duration = end_sec - start_sec
        try:
            y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
        except Exception:
            y = np.zeros(int(self.sr * self.window_sec))
            
        target_len = int(self.sr * self.window_sec)
        if len(y) > target_len:
            y = y[:target_len]
        elif len(y) < target_len:
            pad_len = target_len - len(y)
            y = np.pad(y, (0, pad_len), 'constant')
            
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length, n_mels=self.n_mels
        )
        mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-6)
        
        mel_tensor = torch.tensor(mel_spec, dtype=torch.float32).unsqueeze(0)
        if self.is_train and self.augment:
            mel_tensor = self.freq_mask(mel_tensor)
            mel_tensor = self.time_mask(mel_tensor)
            
        return mel_tensor, torch.tensor(label, dtype=torch.long)

# ==========================================
# 3. 데이터 로딩 & DataLoader 생성 (num_workers=0)
# ==========================================
TRAIN_DIR = r"C:\Users\user\Desktop\DCC\data\train"
VAL_DIR   = r"C:\Users\user\Desktop\DCC\data\val"

print("📦 1/3. 데이터셋 파싱을 시작합니다...")
train_ds = LocalSpeechDataset(TRAIN_DIR, max_files=None, is_train=True, augment=True) 
val_ds   = LocalSpeechDataset(VAL_DIR, max_files=None, is_train=False, augment=False) 

# ⭐️ num_workers=0 으로 윈도우 프리징/멈춤 완벽 방지
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, pin_memory=True) 
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True) 

# ==========================================
# 4. 모델 및 학습 설정 (FP16 AMP 가속)
# ==========================================
print("⚙️ 2/3. 모델 및 GPU 가속기 세팅 중...")
os.makedirs("./checkpoints", exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AudioResNet50(pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.cuda.amp.GradScaler()

# ==========================================
# 5. 실시간 진행률(tqdm) 탑재 훈련 루프
# ==========================================
best_acc, best_f1 = 0.0, 0.0
print(f"🚀 3/3. [AudioResNet-50] 10 에포크 풀학습 시작! (GPU: {device})")

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    
    # ⭐️ 실시간 로딩 바 장착 (1초마다 진행 상황 갱신)
    pbar = tqdm(train_loader, desc=f"[Ep {epoch:02d}/{epochs:02d} Train]")
    for bx, by in pbar:
        bx, by = bx.to(device), by.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = model(bx)
            loss = criterion(out, by)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    scheduler.step()
    
    # 검증(Validation)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for bx, by in tqdm(val_loader, desc=f"[Ep {epoch:02d}/{epochs:02d} Val]"):
            bx = bx.to(device)
            with torch.cuda.amp.autocast():
                prob = torch.sigmoid(model(bx)).squeeze(-1).cpu().numpy()
            all_preds.extend((prob >= 0.5).astype(int))
            all_labels.extend(by.numpy().astype(int))
            
    acc = accuracy_score(all_labels, all_preds) * 100.0
    f1 = f1_score(all_labels, all_preds, average="macro")
    tr_loss = total_loss / len(train_loader)
    print(f"\n📢 [Ep {epoch:02d}/{epochs:02d}] Tr Loss: {tr_loss:.4f} | Val Acc: {acc:.2f}% | F1: {f1:.4f}")
    
    if acc > best_acc:
        best_acc = acc
        best_f1 = f1
        save_path = "./checkpoints/best_resnet_full.pt"
        torch.save(model.state_dict(), save_path)
        print(f"  👉 최고 점수 갱신! 가중치 저장 완료: {save_path} ({acc:.2f}%)")

print(f"\n🎉 모든 학습 완료! 최고 정확도: {best_acc:.2f}% | Macro F1: {best_f1:.4f}")


📦 1/3. 데이터셋 파싱을 시작합니다...
👉 [train] 로드 완료: 873,137개 발화 구간
👉 [val] 로드 완료: 111,919개 발화 구간
⚙️ 2/3. 모델 및 GPU 가속기 세팅 중...


C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:146: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


🚀 3/3. [AudioResNet-50] 10 에포크 풀학습 시작! (GPU: cuda)


[Ep 01/10 Train]:   0%|          | 0/27286 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
c:\Users\user\Desktop\DCC\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
c:\Users\user\Desktop\DCC\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(pa


📢 [Ep 01/10] Tr Loss: 0.6937 | Val Acc: 48.10% | F1: 0.3248
  👉 최고 점수 갱신! 가중치 저장 완료: ./checkpoints/best_resnet_full.pt (48.10%)


...


📢 [Ep 02/10] Tr Loss: 0.6927 | Val Acc: 51.94% | F1: 0.3427
  👉 최고 점수 갱신! 가중치 저장 완료: ./checkpoints/best_resnet_full.pt (51.94%)


[Ep 03/10 Train]:   0%|          | 0/27286 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
c:\Users\user\Desktop\DCC\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:163: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Ep 03/10 Val]:   0%|          | 0/3498 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:180: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args.


📢 [Ep 03/10] Tr Loss: 0.6925 | Val Acc: 51.94% | F1: 0.3427


[Ep 04/10 Train]:   0%|          | 0/27286 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:97: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=duration)
c:\Users\user\Desktop\DCC\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\user\AppData\Local\Temp\ipykernel_1664\2423614616.py:163: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Ep 04/10 Train]:  41%|████      | 11183/27286 [1:26:35<2:04:41,  2.15it/s, loss=0.6910]


KeyboardInterrupt: 